In [23]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import StandardScaler

In [4]:
PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "../data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data directory: {DATA_DIR}")

Project root: /Users/tolulopesoetan/Desktop/pl_prediction_model/notebooks
Processed data directory: /Users/tolulopesoetan/Desktop/pl_prediction_model/notebooks/../data/processed


In [7]:
# load dataset
urls = {
    "2223": "https://www.football-data.co.uk/mmz4281/2223/E0.csv",
    "2324": "https://www.football-data.co.uk/mmz4281/2324/E0.csv",
    "2425": "https://www.football-data.co.uk/mmz4281/2425/E0.csv",
}

frames = []

for season, url in urls.items():
    df = pd.read_csv(url)
    df["season"] = season
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)

print("Raw shape:", raw.shape)
print("\nColumns:")
print(raw.columns.tolist())

Raw shape: (1140, 133)

Columns:
['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH', 'IWD', 'IWA', 'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 'VCH', 'VCD', 'VCA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'AHh', 'B365AHH', 'B365AHA', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'B365CH', 'B365CD', 'B365CA', 'BWCH', 'BWCD', 'BWCA', 'IWCH', 'IWCD', 'IWCA', 'PSCH', 'PSCD', 'PSCA', 'WHCH', 'WHCD', 'WHCA', 'VCCH', 'VCCD', 'VCCA', 'MaxCH', 'MaxCD', 'MaxCA', 'AvgCH', 'AvgCD', 'AvgCA', 'B365C>2.5', 'B365C<2.5', 'PC>2.5', 'PC<2.5', 'MaxC>2.5', 'MaxC<2.5', 'AvgC>2.5', 'AvgC<2.5', 'AHCh', 'B365CAHH', 'B365CAHA', 'PCAHH', 'PCAHA', 'MaxCAHH', 'MaxCAHA', 'AvgCAHH', 'AvgCAHA', 'season', 'BFH', 'BFD', 'BFA', '1XBH', '1XBD'

In [8]:
raw[
    [
        "Date",
        "HomeTeam",
        "AwayTeam",
        "FTR",
        "FTHG",
        "FTAG",
        "HTHG",
        "HTAG",
        "B365H",
        "B365D",
        "B365A",
    ]
].head(10)


,Date,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HTHG,HTAG,B365H,B365D,B365A
0,05/08/2022,Crystal Palace,Arsenal,A,0,2,0,1,4.20,3.60,1.85
1,06/08/2022,Fulham,Liverpool,D,2,2,1,0,11.00,6.00,1.25
2,06/08/2022,Bournemouth,Aston Villa,H,2,0,1,0,3.75,3.50,2.00
3,06/08/2022,Leeds,Wolves,H,2,1,1,1,2.25,3.40,3.20
4,06/08/2022,Newcastle,Nott'm Forest,H,2,0,0,0,1.66,3.80,5.25
5,06/08/2022,Tottenham,Southampton,H,4,1,2,1,1.33,5.50,8.50
6,06/08/2022,Everton,Chelsea,A,0,1,0,1,5.50,4.00,1.61
7,07/08/2022,Leicester,Brentford,D,2,2,1,0,2.00,3.75,3.50
8,07/08/2022,Man United,Brighton,A,1,2,0,2,1.60,4.20,5.25
9,07/08/2022,West Ham,Man City,A,0,2,0,1,8.00,5.00,1.36


In [ ]:
# check for missing values
cols_to_check = [
    "FTR",
    "FTHG",
    "FTAG",
    "HTHG",
    "HTAG",
    "B365H",
    "B365D",
    "B365A",
    "HS",
    "AS",
    "HST",
    "AST",
]

raw[cols_to_check].isnull().sum()

FTR      0
FTHG     0
FTAG     0
HTHG     0
HTAG     0
B365H    0
B365D    0
B365A    0
HS       0
AS       0
HST      0
AST      0
dtype: int64

In [10]:
print("Full-time result distribution:")
print(raw["FTR"].value_counts(dropna=False))

print("\nRows per season:")
print(raw.groupby("season")["FTR"].count())

Full-time result distribution:
FTR
H    514
A    364
D    262
Name: count, dtype: int64

Rows per season:
season
2223    380
2324    380
2425    380
Name: FTR, dtype: int64


In [ ]:
# clean rows and parse dates

raw["Date"] = pd.to_datetime(raw["Date"], dayfirst=True, errors="coerce")

required_cols = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTR",
    "FTHG",
    "FTAG",
    "HTHG",
    "HTAG",
    "B365H",
    "B365D",
    "B365A",
]

raw = (
    raw.dropna(subset=required_cols)
       .sort_values("Date")
       .reset_index(drop=True)
       .copy()
)

print("Cleaned shape:", raw.shape)
print("Date range:", raw["Date"].min(), "to", raw["Date"].max())

Cleaned shape: (1140, 133)
Date range: 2022-08-05 00:00:00 to 2025-05-25 00:00:00


In [15]:
# build bookmaker market probabilities

raw["mkt_h"] = 1 / raw["B365H"]
raw["mkt_d"] = 1 / raw["B365D"]
raw["mkt_a"] = 1 / raw["B365A"]

margin = raw["mkt_h"] + raw["mkt_d"] + raw["mkt_a"]

raw["market_home_prob"] = raw["mkt_h"] / margin
raw["market_draw_prob"] = raw["mkt_d"] / margin
raw["market_away_prob"] = raw["mkt_a"] / margin

#raw

raw["market_prediction"] = (
    raw[["market_home_prob", "market_draw_prob", "market_away_prob"]]
    .idxmax(axis=1)
    .map(
        {
            "market_home_prob": "H",
            "market_draw_prob": "D",
            "market_away_prob": "A",
        }
    )
)

market_acc = (raw["market_prediction"] == raw["FTR"]).mean()

print(f"Market baseline accuracy: {market_acc:.3f}")
print("\nMarket prediction counts:")
print(raw["market_prediction"].value_counts())

Market baseline accuracy: 0.565

Market prediction counts:
market_prediction
H    722
A    418
Name: count, dtype: int64


In [16]:
raw[
    [
        "Date",
        "HomeTeam",
        "AwayTeam",
        "FTR",
        "market_home_prob",
        "market_draw_prob",
        "market_away_prob",
        "market_prediction",
        "season",
    ]
].head(10)

,Date,HomeTeam,AwayTeam,FTR,market_home_prob,market_draw_prob,market_away_prob,market_prediction,season
0,2022-08-05,Crystal Palace,Arsenal,A,0.225381,0.262944,0.511675,A,2223
1,2022-08-06,Fulham,Liverpool,D,0.085960,0.157593,0.756447,A,2223
2,2022-08-06,Bournemouth,Aston Villa,H,0.253394,0.271493,0.475113,A,2223
3,2022-08-06,Leeds,Wolves,H,0.422853,0.279829,0.297318,H,2223
4,2022-08-06,Newcastle,Nott'm Forest,H,0.570440,0.249192,0.180368,H,2223
5,2022-08-06,Tottenham,Southampton,H,0.715160,0.172939,0.111901,H,2223
6,2022-08-06,Everton,Chelsea,A,0.172677,0.237431,0.589891,A,2223
7,2022-08-07,Leicester,Brentford,D,0.475113,0.253394,0.271493,H,2223
8,2022-08-07,Man United,Brighton,A,0.593220,0.225989,0.180791,H,2223
9,2022-08-07,West Ham,Man City,A,0.117892,0.188627,0.693481,A,2223


In [17]:
# save processed data

output_path = DATA_DIR / "raw_matches.csv"
raw.to_csv(output_path, index=False)

print(f"Saved cleaned matches to: {output_path}")

Saved cleaned matches to: /Users/tolulopesoetan/Desktop/pl_prediction_model/notebooks/../data/processed/raw_matches.csv


In [19]:
# create rolling form creatures

def compute_rolling_features(df, n_matches=5):
    """
    Create rolling form features that use past matches to calculate average points, goals scored, 
    goals conceded and recent home team win rate vs this opponent
    """
    df = df.copy().sort_values("Date").reset_index(drop=True)

    team_history = {}
    rows = []

    def get_form(history, n=n_matches):
        """
        Calculate recent form statistics from a team's previous matches. 
        Returns average points, goals scored, goals conceded, and number of games over the last n matches. 
        If there is no history yet, returns None values.
        """
        recent = history[-n:]
        if len(recent) == 0:
            return {
                "form_pts": None,
                "goals_for": None,
                "goals_against": None,
                "n_games": 0,
            }

        return {
            "form_pts": sum(match["pts"] for match in recent) / len(recent),
            "goals_for": sum(match["gf"] for match in recent) / len(recent),
            "goals_against": sum(match["ga"] for match in recent) / len(recent),
            "n_games": len(recent),
        }

    def get_h2h_home_winrate(home_team, away_team, home_history, n=n_matches):
        """
        Calculate the recent head-to-head home win rate for the home team against the away team, 
        based only on the home team's stored history. 
        If no recent head-to-head history exists, return 0.33 as a neutral default.
        """
        recent_h2h = [match for match in home_history if match["opponent"] == away_team][-n:]
        if len(recent_h2h) == 0:
            return 0.33
        wins = sum(1 for match in recent_h2h if match["pts"] == 3)
        return wins / len(recent_h2h)

    for _, row in df.iterrows():
        home = row["HomeTeam"]
        away = row["AwayTeam"]
        date = row["Date"]

        home_history = team_history.get(home, [])
        away_history = team_history.get(away, [])

        home_form = get_form(home_history)
        away_form = get_form(away_history)
        h2h_home_winrate = get_h2h_home_winrate(home, away, home_history)

        rows.append(
            {
                "Date": date,
                "HomeTeam": home,
                "AwayTeam": away,
                "FTR": row["FTR"],
                "FTHG": row["FTHG"],
                "FTAG": row["FTAG"],
                "HTHG": row["HTHG"],
                "HTAG": row["HTAG"],
                "home_form_pts": home_form["form_pts"],
                "home_goals_for": home_form["goals_for"],
                "home_goals_against": home_form["goals_against"],
                "away_form_pts": away_form["form_pts"],
                "away_goals_for": away_form["goals_for"],
                "away_goals_against": away_form["goals_against"],
                "h2h_home_winrate": h2h_home_winrate,
                "market_home_prob": row["market_home_prob"],
                "market_draw_prob": row["market_draw_prob"],
                "market_away_prob": row["market_away_prob"],
                "season": row["season"],
            }
        )

        home_pts = 3 if row["FTR"] == "H" else 1 if row["FTR"] == "D" else 0
        away_pts = 3 if row["FTR"] == "A" else 1 if row["FTR"] == "D" else 0

        if home not in team_history:
            team_history[home] = []
        if away not in team_history:
            team_history[away] = []

        team_history[home].append(
            {
                "date": date,
                "pts": home_pts,
                "gf": row["FTHG"],
                "ga": row["FTAG"],
                "opponent": away,
            }
        )

        team_history[away].append(
            {
                "date": date,
                "pts": away_pts,
                "gf": row["FTAG"],
                "ga": row["FTHG"],
                "opponent": home,
            }
        )

    return pd.DataFrame(rows)

In [20]:
features_basic = compute_rolling_features(raw)

features_basic = (
    features_basic.dropna(subset=["home_form_pts", "away_form_pts"])
                  .reset_index(drop=True)
                  .copy()
)

features_basic["market_prediction"] = (
    features_basic[["market_home_prob", "market_draw_prob", "market_away_prob"]]
    .idxmax(axis=1)
    .map(
        {
            "market_home_prob": "H",
            "market_draw_prob": "D",
            "market_away_prob": "A",
        }
    )
)

print("Engineered feature shape:", features_basic.shape)
print("\nRows per season after feature engineering:")
print(features_basic.groupby("season").size())

Engineered feature shape: (1126, 20)

Rows per season after feature engineering:
season
2223    370
2324    377
2425    379
dtype: int64


In [21]:
features_basic[
    [
        "Date",
        "HomeTeam",
        "AwayTeam",
        "FTR",
        "home_form_pts",
        "home_goals_for",
        "home_goals_against",
        "away_form_pts",
        "away_goals_for",
        "away_goals_against",
        "h2h_home_winrate",
        "market_home_prob",
        "market_draw_prob",
        "market_away_prob",
        "season",
    ]
].head(10)

,Date,HomeTeam,AwayTeam,FTR,home_form_pts,home_goals_for,home_goals_against,away_form_pts,away_goals_for,away_goals_against,h2h_home_winrate,market_home_prob,market_draw_prob,market_away_prob,season
0,2022-08-13,Aston Villa,Everton,H,0.0,0.0,2.0,0.0,0.0,1.0,0.33,0.528197,0.271644,0.200159,2223
1,2022-08-13,Arsenal,Leicester,H,3.0,2.0,0.0,1.0,2.0,2.0,0.33,0.617499,0.218193,0.164308,2223
2,2022-08-13,Brentford,Man United,H,1.0,2.0,2.0,0.0,1.0,2.0,0.33,0.253394,0.271493,0.475113,2223
3,2022-08-13,Brighton,Newcastle,D,3.0,2.0,1.0,3.0,2.0,0.0,0.33,0.387253,0.296491,0.316257,2223
4,2022-08-13,Southampton,Leeds,D,0.0,1.0,4.0,3.0,2.0,1.0,0.33,0.422179,0.271401,0.306420,2223
5,2022-08-13,Wolves,Fulham,D,0.0,1.0,2.0,1.0,2.0,2.0,0.33,0.413955,0.288514,0.297530,2223
6,2022-08-13,Man City,Bournemouth,H,3.0,2.0,0.0,3.0,2.0,0.0,0.33,0.894382,0.072927,0.032691,2223
7,2022-08-14,Nott'm Forest,West Ham,H,0.0,0.0,2.0,0.0,0.0,2.0,0.33,0.225381,0.262944,0.511675,2223
8,2022-08-14,Chelsea,Tottenham,D,3.0,1.0,0.0,3.0,4.0,1.0,0.33,0.422853,0.279829,0.297318,2223
9,2022-08-15,Liverpool,Crystal Palace,D,1.0,2.0,2.0,0.0,0.0,2.0,0.33,0.800920,0.140013,0.059068,2223


In [24]:
# logistic regression to check the new features

FEATURES_BASIC = [
    "home_form_pts",
    "home_goals_for",
    "home_goals_against",
    "away_form_pts",
    "away_goals_for",
    "away_goals_against",
    "h2h_home_winrate",
    "market_home_prob",
    "market_draw_prob",
]

train = features_basic[features_basic["season"].isin(["2223", "2324"])].copy()
test = features_basic[features_basic["season"] == "2425"].copy()

X_train = train[FEATURES_BASIC]
y_train = train["FTR"]
X_test = test[FEATURES_BASIC]
y_test = test["FTR"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(class_weight="balanced", max_iter=1000)
lr.fit(X_train_scaled, y_train)

lr_preds = lr.predict(X_test_scaled)

print(f"Market baseline accuracy: {accuracy_score(y_test, test['market_prediction']):.3f}")
print(f"LR accuracy:             {accuracy_score(y_test, lr_preds):.3f}")
print(f"LR weighted F1:          {f1_score(y_test, lr_preds, average='weighted'):.3f}")
print()
print(classification_report(y_test, lr_preds, zero_division=0))

Market baseline accuracy: 0.538
LR accuracy:             0.470
LR weighted F1:          0.471

              precision    recall  f1-score   support

           A       0.54      0.50      0.52       131
           D       0.20      0.22      0.21        93
           H       0.58      0.59      0.59       155

    accuracy                           0.47       379
   macro avg       0.44      0.44      0.44       379
weighted avg       0.47      0.47      0.47       379



In [25]:
features_output_path = DATA_DIR / "features_basic.csv"
features_basic.to_csv(features_output_path, index=False)

print(f"Saved engineered features to: {features_output_path}")

Saved engineered features to: /Users/tolulopesoetan/Desktop/pl_prediction_model/notebooks/../data/processed/features_basic.csv
